# Stage 14B: extended residual correction and absolute-tail gate

Fast CPU audit. It reuses Stage 14 cross-fitted raw residuals, extends weight/cap to the boundary, and replaces worst-tail share with absolute SSE/CVaR/P90/max gates. No model is retrained and no submission is created.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)


## Reuse Stage 14

The Stage 14 OOF files contain generic and stacked residual predictions for every outer fold.


In [ ]:
STAGE14_RUN_ID = 'stage14_crossfit_emission_residual_full_v001'
stage14_run = artifact_dir / STAGE14_RUN_ID
assert (stage14_run / 'gate_summary.json').is_file(), 'Run notebook 310 first'
stage14_summary = json.loads((stage14_run / 'gate_summary.json').read_text())
assert stage14_summary['robust_generic_spec'] == 'generic_w050_cap8'
print('Stage 14 boundary result verified')


## Run extended nested gate

Keep `LIMIT_WELLS = None`. This is a parquet-only CPU calculation and should finish quickly.


In [ ]:
RUN_ID = 'stage14b_extended_residual_gate_full_v001'
LIMIT_WELLS = None
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    command = [
        'uv', 'run', 'rogii-emission-residual-gate',
        '--config', 'configs/experiment/stage14b_extended_residual_gate.yaml',
        '--stage14-run', str(stage14_run), '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
    ]
    if LIMIT_WELLS is not None:
        command += ['--limit-wells', str(LIMIT_WELLS)]
    subprocess.run(command, cwd=repo_dir, check=True)
else:
    print('Reusing completed run:', run_dir)


## Decision

Absolute tail deltas must be non-positive. A positive worst-tail share alone is not a failure when absolute worst-well loss falls.


In [ ]:
summary = json.loads((run_dir / 'gate_summary.json').read_text(encoding='utf-8'))
families = summary['family_reports']
decision = {
    'promoted_to_full_residual_training': summary['promoted_to_full_residual_training'],
    'standard_base_rmse': families['fold']['base_metrics']['pooled_rmse'], 'standard_nested_rmse': families['fold']['nested_metrics']['pooled_rmse'], 'standard_delta': families['fold']['nested_rmse_delta'],
    'spatial_base_rmse': families['spatial_fold']['base_metrics']['pooled_rmse'], 'spatial_nested_rmse': families['spatial_fold']['nested_metrics']['pooled_rmse'], 'spatial_delta': families['spatial_fold']['nested_rmse_delta'],
    'typewell_base_rmse': families['typewell_fold']['base_metrics']['pooled_rmse'], 'typewell_nested_rmse': families['typewell_fold']['nested_metrics']['pooled_rmse'], 'typewell_delta': families['typewell_fold']['nested_rmse_delta'],
    'bootstrap_95pct': {name: [report['bootstrap']['ci_2_5'], report['bootstrap']['ci_97_5']] for name, report in families.items()},
    'standard_absolute_tail_delta': summary['standard_absolute_tail_delta'],
    'robust_extended_generic_spec': summary['robust_extended_generic_spec'],
    'gates': summary['gates'], 'next_step': summary['next_step'],
}
decision


In [ ]:
import pandas as pd
rows = []
for family, report in families.items():
    for name, values in report['profile_reports'].items():
        tail = values['candidate_tail']; base_tail = values['base_tail']
        rows.append({'family': family, 'profile': name, 'rmse_delta': values['pooled_rmse_delta'], 'worst_fold_delta': max(values['fold_deltas'].values()), 'worst_tail_sse_delta': tail['worst_tail_sse'] - base_tail['worst_tail_sse'], 'cvar_delta': tail['well_rmse_cvar'] - base_tail['well_rmse_cvar'], 'p90_delta': tail['well_rmse_p90'] - base_tail['well_rmse_p90'], 'well_max_delta': tail['well_rmse_max'] - base_tail['well_rmse_max']})
pd.DataFrame(rows).sort_values(['family', 'rmse_delta']).reset_index(drop=True)


## Send back

Send the decision dictionary and extended profile table. No Kaggle submission is produced.
